# 또 다른 나의 세계: 평행 시나리오 생성과 감성 분석

In [ ]:
!pip install -U transformers
!pip install torch
!pip install sentencepiece
!pip install kobert-transformers
!pip install keybert
!pip install konlpy
# accelerate: "가속 및 자동 배치 도구"입니다. 모델이 GPU 메모리를 효율적으로 쓰게 해주고, device_map="auto" 같은 편리한 기능을 쓸 수 있게 해줍니다.


In [ ]:
!pip install -U huggingface_hub transformers accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from keybert import KeyBERT
from konlpy.tag import Okt
import torch

### 평행세계(parallel_world) 이야기


**또 다른 나의 세계 만들고 평가하기**
- 스토리 생성
- 스토리 요약   
- 텍스트 분석
   - 감정 분석
   - 키워드 추출
   - 주제 분류


#### 평행세계(parallel_world) 스토리 생성

In [ ]:
!rm -rf /root/.cache/huggingface/hub/*

In [ ]:
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"사용 중인 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
model_name = 'MLP-KTLim/llama-3-Korean-Bllossom-8B'

tokenizer = AutoTokenizer.from_pretrained(model_name)

gen_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,device_map="auto")
generator = pipeline("text-generation",model=gen_model,tokenizer=tokenizer,
    max_new_tokens=200,do_sample=True,temperature=0.8,top_p=0.9, repetition_penalty=1.2)


In [ ]:
story = generator('나는 닌자 세계에 살고 있는 데이터분석적 사고를 가진 닌자야, 이제 임무를 떠나려고해')[0]['generated_text']
story

In [ ]:
import re

clean_story = re.sub('<.*?>', '', story)
clean_story = clean_story.replace("\n", " ")
clean_story

#### 평행세계 스토리 요약

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "digit82/kobart-summarization"

summary_tokenizer = AutoTokenizer.from_pretrained(model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
summary_model.to(device)
inputs = summary_tokenizer(clean_story, return_tensors="pt", max_length=80, truncation=True)
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
summary_ids = summary_model.generate(input_ids=input_ids,attention_mask=attention_mask,max_length=150,
    min_length=50,num_beams=5, no_repeat_ngram_size=3,early_stopping=True)


In [ ]:
summary = summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

#### 나의 평행세계는 긍정적인가? 부정적인가?


In [ ]:
# sentiment = pipeline('sentiment-analysis', model = 'beomi/KcELECTRA-base-v2022' )
# senti_story = sentiment(clean_story)

# print('나의 또 다른 세계의 이야기는 긍정적인가? 부정적인가? --> ', senti_story) # 약간 긍정적

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

model_name = "beomi/KcELECTRA-base-v2022"

sent_tokenizer = AutoTokenizer.from_pretrained(model_name)
sent_model = AutoModelForSequenceClassification.from_pretrained(model_name)
inputs = sent_tokenizer(clean_story, return_tensors="pt", truncation=True, max_length=200)

# 추론
with torch.no_grad():
    outputs = sent_model(**inputs)

# softmax로 확률 계산
probs = F.softmax(outputs.logits, dim=-1)

# 출력
print(f"Negative: {probs[0][0].item():.4f}")
print(f"Positive: {probs[0][1].item():.4f}")

# label 결정
label = "Positive" if probs[0][1] > probs[0][0] else "Negative"
print("전체 텍스트 감성 :", label)


print('나의 또 다른 세계의 이야기는 긍정적인가? 부정적인가? --> ', label) # 약간 긍정적



#### 평행 세계의 핵심 키워드

In [ ]:
key_word_extraction = KeyBERT()
okt = Okt()

nouns = okt.nouns(clean_story)
sententce = ' '.join(nouns)
keywords = key_word_extraction.extract_keywords(sententce, top_n = 10)
print(keywords)

##또 다른 나의 세상의 모습을 확인해보자!
- ** 주의사항... 위의 모델을 학습시켜놔야함..(오래걸림..)

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

parallel_world_sentences = [
    "나는 닌자 세계에 살고 있는 데이터분석적 사고를 가진 닌자야, 이제 임무를 떠나려고해, 나는 인술, 환술, 체술 모두 능숙하지.",
    "나는 마법 왕국에서 데이터로 미래를 예측하는 마법사야, 이제 숨겨진 알고리즘의 탑을 향해 마법을 수련하고 있어.",
    "나는 우주 제국의 자유로운 항해사야, 은하의 패턴을 분석하며 새로운 별지도를 찾으러 여행을 시작하려고 해.",
    "나는 사이버 도시에서 데이터를 해킹해 진실을 찾는 분석가 요원이야, 이제 거대한 네트워크 미궁 속으로 들어가려 해.",
    "나는 고대 용들이 사는 세계에서 천리안을 가진 전술가야, 이제 전설의 드래곤 계곡에 가서 드래곤들과 싸움을 시작할거야.",
    "나는 시간 여행자야, 역사 속 숨겨진 패턴을 찾기 위해 과거와 미래를 넘나들어서 세계정복을 시작하려 해."
]

def parallel_world_story(start_sententce):
    # 패러럴월드 스토리 생성
    story = generator(start_sententce)[0]['generated_text']
    clean_story = re.sub('<.*?>', '', story)
    clean_story = clean_story.replace("\n", " ")
    print('1. 나의 또 다른 세계의 이야기는 무엇일까? -->', clean_story)

    # 요약 생성
    summary_model.to(device)
    inputs = summary_tokenizer(clean_story, return_tensors="pt", max_length=80, truncation=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    summary_ids = summary_model.generate(input_ids=input_ids,attention_mask=attention_mask,max_length=150,
        min_length=50,num_beams=5, no_repeat_ngram_size=3,early_stopping=True)

    summary = summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print('2. 나의 또 다른 세계의 이야기를 요약하자면? -->', summary)



    # 긍부정
    inputs = sent_tokenizer(clean_story, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = sent_model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1)
    print(f"Negative: {probs[0][0].item():.4f} / Positive: {probs[0][1].item():.4f}")
    label = "Positive" if probs[0][1] > probs[0][0] else "Negative"
    print("전체 평행세계 시나리오 감성 :", label)
    print('3. 나의 또 다른 세계의 이야기는 긍정적인가? 부정적인가? --> ', label)

    # 핵심 주제
    nouns = okt.nouns(clean_story)
    sententce = ' '.join(nouns)
    keywords = key_word_extraction.extract_keywords(sententce, top_n = 10)
    print('4. 나의 또 다른 세계의 이야기의 핵심 키워드는? --> ', keywords )

    print()
    print('*'* 200)
    print('스토리 완료')
    print('*'* 200)
    print()
    return {"1. 나의 평행 세계 Story 생성" : clean_story,
            "2. 나의 평행 세계 요약" : summary,
            "3. 나의 평행 세계의 긍부정 상황" : label,
            "4. 나의 평행 세계의 핵심 키워드" : keywords}






In [ ]:
parallel_world_story_list = []
for another_story in parallel_world_sentences:
    story_list = parallel_world_story(another_story)
    parallel_world_story_list.append(story_list)